In [1]:
import json
import re
import numpy as np
import pandas as pd
import torch

from transformers import pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
apartments_df = pd.read_csv('apartments_data-1.csv')

### Loading in Qwen Model  

In [3]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

if torch.cuda.is_available():
    device = "cuda"
    model_kwargs = {"dtype": torch.float16}
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
    model_kwargs = {"dtype": torch.float16}
else:
    device = "cpu"
    model_kwargs = {"dtype": torch.float32}

print(f"Loading {model_name} on {device}...")
generator = pipeline(
    "text-generation",
    model=model_name,
    device=device,
    **model_kwargs
)
print("Model loaded.")


Loading Qwen/Qwen2.5-1.5B-Instruct on mps...


Device set to use mps


Model loaded.


### Adding in Hard Filter Schema

In [4]:
BINARY_FILTER_COLUMNS = [
    "in_unit_laundry",
    "dishwasher",
    "central_air",
    "parking_included",
    "gym_in_building",
    "balcony",
    "pets_allowed",
    "heat_included",
    "water_included"
]

NUMERIC_FILTER_COLUMNS = [
    "rent_max",
    "bedrooms_min",
    "bathrooms_min",
    "sqft_min"
]

OTHER_FILTER_COLUMNS = [
    "neighborhood"
]

### Parsing and Normalization

In [5]:
def extract_json_from_text(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in model output.")
    return json.loads(match.group(0))


def normalize_binary(value):
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, (int, float)):
        return 1 if value >= 1 else 0
    if isinstance(value, str):
        return 1 if value.strip().lower() in {"1", "true", "yes"} else 0
    return 0


def normalize_number(value):
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return float(value)
    if isinstance(value, str):
        cleaned = re.sub(r"[^\d.]", "", value)
        if cleaned == "":
            return None
        return float(cleaned)
    return None


def normalize_neighborhood(value):
    if value is None:
        return []

    if isinstance(value, str):
        val = value.strip()
        return [val] if val else []

    if isinstance(value, list):
        cleaned = []
        for v in value:
            if isinstance(v, str) and v.strip():
                cleaned.append(v.strip())
        return cleaned

    return []


def normalize_filter_output(raw_output):
    normalized = {}

    for key in BINARY_FILTER_COLUMNS:
        normalized[key] = normalize_binary(raw_output.get(key, 0))

    normalized["rent_max"] = normalize_number(raw_output.get("rent_max"))
    normalized["bedrooms_min"] = normalize_number(raw_output.get("bedrooms_min"))
    normalized["bathrooms_min"] = normalize_number(raw_output.get("bathrooms_min"))
    normalized["sqft_min"] = normalize_number(raw_output.get("sqft_min"))

    normalized["neighborhood"] = normalize_neighborhood(raw_output.get("neighborhood"))

    return normalized

### LLM Extraction



In [6]:
def extract_filters_llm(user_text, generator, max_new_tokens=300):
    prompt = f"""
You are an information extraction system for apartment search.

Read the user's apartment description and return ONLY a valid JSON object with exactly these keys:

- in_unit_laundry
- dishwasher
- central_air
- parking_included
- gym_in_building
- balcony
- pets_allowed
- heat_included
- water_included
- neighborhood
- rent_max
- bedrooms_min
- bathrooms_min
- sqft_min

Rules:
- For the amenity keys, return 1 if clearly requested, otherwise 0.
- neighborhood should be a JSON list of desired neighborhoods. If none are mentioned, return [].
- rent_max should be the maximum rent the user wants to pay. If not mentioned, return null.
- bedrooms_min should be the minimum number of bedrooms requested. If not mentioned, return null.
- bathrooms_min should be the minimum number of bathrooms requested. If not mentioned, return null.
- sqft_min should be the minimum square footage requested. If not mentioned, return null.
- Do not include explanations.
- Output JSON only.

User description:
\"\"\"{user_text}\"\"\"
"""

    response = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        return_full_text=False
    )

    raw_text = response[0]["generated_text"].strip()
    parsed = extract_json_from_text(raw_text)
    return normalize_filter_output(parsed)


def safe_extract_filters_llm(user_text, generator):
    try:
        return extract_filters_llm(user_text, generator)
    except Exception:
        fallback = {key: 0 for key in BINARY_FILTER_COLUMNS}
        fallback.update({
            "neighborhood": [],
            "rent_max": None,
            "bedrooms_min": None,
            "bathrooms_min": None,
            "sqft_min": None
        })
        return fallback

### Apply Filters

In [7]:
def apply_all_filters(apartments_df, filters_dict):
    df = apartments_df.copy()

    # binary amenity filters
    for col in BINARY_FILTER_COLUMNS:
        if filters_dict[col] == 1:
            df = df[df[col] == 1]

    # neighborhood filter
    desired_neighborhoods = filters_dict["neighborhood"]
    if desired_neighborhoods:
        desired_lower = {n.lower() for n in desired_neighborhoods}
        df = df[df["neighborhood"].astype(str).str.lower().isin(desired_lower)]

    # numeric filters
    if filters_dict["rent_max"] is not None:
        df = df[df["rent"] <= filters_dict["rent_max"]]

    if filters_dict["bedrooms_min"] is not None:
        df = df[df["bedrooms"] >= filters_dict["bedrooms_min"]]

    if filters_dict["bathrooms_min"] is not None:
        df = df[df["bathrooms"] >= filters_dict["bathrooms_min"]]

    if filters_dict["sqft_min"] is not None:
        df = df[df["sqft"] >= filters_dict["sqft_min"]]

    return df

### Text Prep

In [8]:
def combine_listing_text(df):
    df = df.copy()

    text_cols = ["listing_description", "review_1", "review_2", "review_3"]
    for col in text_cols:
        if col not in df.columns:
            df[col] = ""

    df[text_cols] = df[text_cols].fillna("").astype(str)

    df["combined_text"] = (
        df["listing_description"] + " " +
        df["listing_description"] + " " +
        df["review_1"] + " " +
        df["review_2"] + " " +
        df["review_3"]
    ).str.strip()

    return df


def min_max_scale(series):
    series = series.astype(float)
    if series.max() == series.min():
        return pd.Series(np.zeros(len(series)), index=series.index)
    return (series - series.min()) / (series.max() - series.min())

### Second Pass Rankings

In [9]:
def rank_apartments_second_pass(
    filtered_df,
    user_text,
    top_n=10,
    similarity_threshold=0.08,
    similarity_weight=0.85,
    ctr_weight=0.15
):
    if filtered_df.empty:
        return pd.DataFrame(), pd.DataFrame()

    df = combine_listing_text(filtered_df)

    corpus = df["combined_text"].tolist() + [user_text]

    vectorizer = TfidfVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        max_features=5000
    )

    tfidf_matrix = vectorizer.fit_transform(corpus)
    apt_matrix = tfidf_matrix[:-1]
    user_vector = tfidf_matrix[-1]

    df["text_similarity"] = cosine_similarity(apt_matrix, user_vector).flatten()
    df["click_through_rate"] = df["click_through_rate"].fillna(0).astype(float)
    df["ctr_norm"] = min_max_scale(df["click_through_rate"])

    df["final_score"] = (
        similarity_weight * df["text_similarity"] +
        ctr_weight * df["ctr_norm"]
    )

    strong_matches = df[df["text_similarity"] >= similarity_threshold].copy()
    strong_matches = strong_matches.sort_values("final_score", ascending=False)

    if len(strong_matches) >= top_n:
        recommended = strong_matches.head(top_n).copy()
    else:
        recommended_ids = strong_matches["listing_id"].tolist()
        fillers = df[~df["listing_id"].isin(recommended_ids)].copy()
        fillers = fillers.sort_values("final_score", ascending=False)
        recommended = pd.concat(
            [strong_matches, fillers.head(top_n - len(strong_matches))],
            ignore_index=True
        )

    other_apartments = df[~df["listing_id"].isin(recommended["listing_id"])].copy()
    other_apartments = other_apartments.sort_values(
        ["text_similarity", "click_through_rate"],
        ascending=False
    )

    return recommended, other_apartments

### Full Pipeline

In [10]:
def recommend_apartments_llm(apartments_df, user_description, generator, top_n=10):
    extracted_filters = safe_extract_filters_llm(user_description, generator)
    filtered_df = apply_all_filters(apartments_df, extracted_filters)

    top_recs, other_apartments = rank_apartments_second_pass(
        filtered_df=filtered_df,
        user_text=user_description,
        top_n=top_n
    )

    return {
        "extracted_filters": extracted_filters,
        "num_after_first_pass": len(filtered_df),
        "top_10_recommendations": top_recs,
        "other_apartments": other_apartments
    }

### Example Usage

In [11]:
user_description_1 = """
I want a one bedroom apartment in Back Bay or South End for no more than $3200.
I want at least one bathroom and at least 700 square feet.
It must have in-unit laundry, dishwasher, and central air.
A gym in the building would be great too.
"""

user_description_2 = """
I want a one bedroom apartment in any neighborhood for no more than $2000.
I want one bathroom and at least 200 square feet.
It must have dishwasher and central air.
A gym in the building would be great too. It would be great if the apartment has good reviews, and is in a young and hip part of town.
"""

results = recommend_apartments_llm(apartments_df, user_description_1, generator, top_n=10)

print("Extracted filters:")
print(results["extracted_filters"])

print("\nListings remaining after first pass:")
print(results["num_after_first_pass"])

cols_to_show = [
    "listing_id",
    "neighborhood",
    "rent",
    "bedrooms",
    "bathrooms",
    "sqft",
    "click_through_rate",
    "text_similarity",
    "final_score"
]

print("\nTop 10 recommendations:")
try:
    display(results["top_10_recommendations"][cols_to_show])
except KeyError:
    print("No apartments meet this criteria")

print("\nOther apartments:")
try:
    display(results["other_apartments"][cols_to_show].head(20))
except KeyError:
    print("No apartments meet this criteria")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/Users/ariellerabinovich/anaconda3/envs/ds_new/lib/python3.11/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


Extracted filters:
{'in_unit_laundry': 1, 'dishwasher': 1, 'central_air': 1, 'parking_included': 0, 'gym_in_building': 1, 'balcony': 0, 'pets_allowed': 0, 'heat_included': 0, 'water_included': 0, 'rent_max': 3200.0, 'bedrooms_min': 1.0, 'bathrooms_min': 1.0, 'sqft_min': 700.0, 'neighborhood': ['Back Bay', 'South End']}

Listings remaining after first pass:
1

Top 10 recommendations:


,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,click_through_rate,text_similarity,final_score
0,APT-1205,Back Bay,2944,1,1,759,0.0607,0.02699,0.022942



Other apartments:


,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,click_through_rate,text_similarity,final_score


### Compare with traditional way

In [34]:
test_prompt_1 = "I want a one bedroom apartment in  Fenway, Cambridge, or Allston  a studio setup is fine, for no more than 2200 a month. I want 1 bathroom and A/C. \
    I want my apartment to be sunny and near fun restuarants."

test_prompt_2 = "I want a 3 bedroom, 2 bathroom apartment in Back Bay, South Boston, Brighton, or Allston for no more than 5000 a month. I need parking. \
I am living with my friend, and we need to be close to transit to commute. \
I ideally want a modern apartment with an ew, clean aesthetic. I do not need A/C or in-unit laundry or heat/water included"

test_prompt_3 = "I want a 2 bedroom 2 bath in Beacon Hill. I don't need A/C but would prefer somewhere over 650 sqft.\
My roommate and I want to be close to a gym and grocery store. I don't care about furnishings or appliances but \
    I want there to be tons of natural light, as I want it to feel homey. I don't need heat/water included."


### Traditional -- only extracting "hard" features

In [14]:
test_dict_1 = extract_filters_llm(test_prompt_1,generator)

/Users/ariellerabinovich/anaconda3/envs/ds_new/lib/python3.11/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


In [16]:
test_df_1 = apply_all_filters(apartments_df, test_dict_1)

In [17]:
test_df_1

,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,gym_in_building,balcony,pets_allowed,heat_included,water_included,listing_description,review_1,review_2,review_3,click_through_rate
68,APT-1068,Allston,1925,1,1,650,2,1944,1,1,...,0,0,0,0,0,Basic one-bedroom with street parking availabl...,It served its purpose. high ceilings but manag...,The apartment looked better in photos. the wal...,Struggled with heating and cooling issues the ...,0.0565
161,APT-1161,Allston,1682,1,1,628,3,1957,0,1,...,0,0,1,0,1,No-frills one-bedroom in a convenient location...,Not great. heating and cooling issues was neve...,Had ongoing issues with cracks in the walls. N...,One of the better apartments I've rented. in-u...,0.0466
313,APT-1313,Allston,2101,3,3,1253,5,1947,1,1,...,0,0,1,0,1,Nice three-bedroom with a functional layout an...,Below average. the hallways are a bit run-down...,Below average. the elevator is slow and the bu...,Disappointing overall. you can hear the neighb...,0.0410
393,APT-1393,Allston,1775,1,1,544,1,1940,1,1,...,0,0,1,0,0,Updated one-bedroom in a well-managed building...,Decent apartment for the price. hardwood floor...,It served its purpose. renovated bathroom but ...,Solid place to live. building amenities was be...,0.0551
480,APT-1480,Fenway,2029,1,1,708,6,2011,0,1,...,1,1,1,0,1,Cozy one-bedroom with good natural light and h...,Would rent again in a heartbeat. in-unit laund...,Overpriced for what you get. the closets are t...,Average experience overall. renovated bathroom...,0.0649


In [28]:
test_dict_2 = extract_filters_llm(test_prompt_2,generator)
test_df_2 = apply_all_filters(apartments_df, test_dict_2)

/Users/ariellerabinovich/anaconda3/envs/ds_new/lib/python3.11/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


In [30]:
test_df_2

,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,gym_in_building,balcony,pets_allowed,heat_included,water_included,listing_description,review_1,review_2,review_3,click_through_rate
26,APT-1026,Brighton,2754,3,3,1258,2,1949,1,1,...,1,0,1,0,1,Impeccably maintained three-bedroom in a sough...,Really enjoyed my time here. natural light and...,Very happy with this place. closet space is ex...,Very happy with this place. updated appliances...,0.0683
124,APT-1124,Brighton,3011,4,4,1563,3,1952,0,1,...,0,0,1,1,1,Cozy four-bedroom with good natural light and ...,It served its purpose. layout but management c...,Would rent again in a heartbeat. natural light...,Not great. a malfunctioning thermostat was nev...,0.0336
234,APT-1234,Back Bay,4242,3,3,1302,5,1927,0,1,...,1,0,0,1,0,Beautifully renovated three-bedroom with moder...,Super comfortable space. rooftop access and I ...,Great apartment. central air made it feel like...,Had a great experience. gym and it's super wal...,0.0585


In [35]:
test_dict_3 = extract_filters_llm(test_prompt_3,generator)
test_df_3 = apply_all_filters(apartments_df, test_dict_3)

/Users/ariellerabinovich/anaconda3/envs/ds_new/lib/python3.11/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


In [36]:
test_dict_3

{'in_unit_laundry': 0,
 'dishwasher': 1,
 'central_air': 0,
 'parking_included': 0,
 'gym_in_building': 1,
 'balcony': 0,
 'pets_allowed': 0,
 'heat_included': 0,
 'water_included': 0,
 'rent_max': None,
 'bedrooms_min': 2.0,
 'bathrooms_min': 2.0,
 'sqft_min': None,
 'neighborhood': ['Beacon Hill']}

In [37]:
test_df_3

,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,gym_in_building,balcony,pets_allowed,heat_included,water_included,listing_description,review_1,review_2,review_3,click_through_rate
148,APT-1148,Beacon Hill,5134,3,3,1248,4,1933,1,1,...,1,0,0,1,0,Stunning three-bedroom in the heart of Beacon ...,"Clean, well-maintained, and courtyard. No comp...",Solid place to live. modern finishes was bette...,My favorite apartment so far. central air and ...,0.0535
252,APT-1252,Beacon Hill,5414,3,3,1122,1,1928,0,1,...,1,0,0,0,0,Beautifully renovated three-bedroom with moder...,Really appreciated high ceilings. The manageme...,Really enjoyed my time here. central air and t...,Average experience overall. renovated bathroom...,0.0580
276,APT-1276,Beacon Hill,4509,2,2,787,1,1904,0,1,...,1,0,1,0,0,Elegant two-bedroom offering a perfect blend o...,Really enjoyed my time here. rooftop access an...,One of the better apartments I've rented. cent...,Perfect for young professionals. storage space...,0.0390
359,APT-1359,Beacon Hill,3959,2,2,1073,6,1911,1,1,...,1,0,1,0,1,Move-in ready two-bedroom with designer touche...,Lived here for two years and loved it. rooftop...,Would rent again in a heartbeat. hardwood floo...,Adequate for the area. bay windows. Could use ...,0.0786
418,APT-1418,Beacon Hill,3900,2,2,800,3,1920,0,1,...,1,0,1,0,1,Elegant two-bedroom offering a perfect blend o...,The apartment exceeded my expectations. quiet ...,Great apartment. rooftop access made it feel l...,Great apartment. natural light made it feel li...,0.0560


### Running Full Pipeline

In [38]:
results_1 = recommend_apartments_llm(apartments_df, test_prompt_1, generator, top_n=5)


/Users/ariellerabinovich/anaconda3/envs/ds_new/lib/python3.11/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)


In [43]:
test_prompt_1

'I want a one bedroom apartment in  Fenway, Cambridge, or Allston  a studio setup is fine, for no more than 2200 a month. I want 1 bathroom and A/C.     I want my apartment to be sunny and near fun restuarants.'

In [44]:
print("\nTop 10 recommendations:")
try:
    display(results_1["top_10_recommendations"])
except KeyError:
    print("No apartments meet this criteria")

print("\nOther apartments:")
try:
    display(results_1["other_apartments"].head(20))
except KeyError:
    print("No apartments meet this criteria")


Top 10 recommendations:


,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,water_included,listing_description,review_1,review_2,review_3,click_through_rate,combined_text,text_similarity,ctr_norm,final_score
0,APT-1480,Fenway,2029,1,1,708,6,2011,0,1,...,1,Cozy one-bedroom with good natural light and h...,Would rent again in a heartbeat. in-unit laund...,Overpriced for what you get. the closets are t...,Average experience overall. renovated bathroom...,0.0649,Cozy one-bedroom with good natural light and h...,0.011455,1.000000,0.159737
1,APT-1393,Allston,1775,1,1,544,1,1940,1,1,...,0,Updated one-bedroom in a well-managed building...,Decent apartment for the price. hardwood floor...,It served its purpose. renovated bathroom but ...,Solid place to live. building amenities was be...,0.0551,Updated one-bedroom in a well-managed building...,0.042960,0.589958,0.125010
2,APT-1068,Allston,1925,1,1,650,2,1944,1,1,...,0,Basic one-bedroom with street parking availabl...,It served its purpose. high ceilings but manag...,The apartment looked better in photos. the wal...,Struggled with heating and cooling issues the ...,0.0565,Basic one-bedroom with street parking availabl...,0.016135,0.648536,0.110995
3,APT-1161,Allston,1682,1,1,628,3,1957,0,1,...,1,No-frills one-bedroom in a convenient location...,Not great. heating and cooling issues was neve...,Had ongoing issues with cracks in the walls. N...,One of the better apartments I've rented. in-u...,0.0466,No-frills one-bedroom in a convenient location...,0.005684,0.234310,0.039978
4,APT-1313,Allston,2101,3,3,1253,5,1947,1,1,...,1,Nice three-bedroom with a functional layout an...,Below average. the hallways are a bit run-down...,Below average. the elevator is slow and the bu...,Disappointing overall. you can hear the neighb...,0.0410,Nice three-bedroom with a functional layout an...,0.005739,0.000000,0.004878



Other apartments:


,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,water_included,listing_description,review_1,review_2,review_3,click_through_rate,combined_text,text_similarity,ctr_norm,final_score


In [42]:
test_df_1

,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,gym_in_building,balcony,pets_allowed,heat_included,water_included,listing_description,review_1,review_2,review_3,click_through_rate
68,APT-1068,Allston,1925,1,1,650,2,1944,1,1,...,0,0,0,0,0,Basic one-bedroom with street parking availabl...,It served its purpose. high ceilings but manag...,The apartment looked better in photos. the wal...,Struggled with heating and cooling issues the ...,0.0565
161,APT-1161,Allston,1682,1,1,628,3,1957,0,1,...,0,0,1,0,1,No-frills one-bedroom in a convenient location...,Not great. heating and cooling issues was neve...,Had ongoing issues with cracks in the walls. N...,One of the better apartments I've rented. in-u...,0.0466
313,APT-1313,Allston,2101,3,3,1253,5,1947,1,1,...,0,0,1,0,1,Nice three-bedroom with a functional layout an...,Below average. the hallways are a bit run-down...,Below average. the elevator is slow and the bu...,Disappointing overall. you can hear the neighb...,0.0410
393,APT-1393,Allston,1775,1,1,544,1,1940,1,1,...,0,0,1,0,0,Updated one-bedroom in a well-managed building...,Decent apartment for the price. hardwood floor...,It served its purpose. renovated bathroom but ...,Solid place to live. building amenities was be...,0.0551
480,APT-1480,Fenway,2029,1,1,708,6,2011,0,1,...,1,1,1,0,1,Cozy one-bedroom with good natural light and h...,Would rent again in a heartbeat. in-unit laund...,Overpriced for what you get. the closets are t...,Average experience overall. renovated bathroom...,0.0649


For this test prompt, we can see that our recommendation bot recommended the apartment with a sunny, cozy description/review at the top whereas the old way of doing it did not.

In [46]:
test_prompt_2

'I want a 3 bedroom, 2 bathroom apartment in Back Bay, South Boston, Brighton, or Allston for no more than 5000 a month. I need parking. I am living with my friend, and we need to be close to transit to commute. I ideally want a modern apartment with an ew, clean aesthetic. I do not need A/C or in-unit laundry or heat/water included'

In [45]:
results_2 = recommend_apartments_llm(apartments_df, test_prompt_2, generator, top_n=5)
print("\nTop 10 recommendations:")
try:
    display(results_2["top_10_recommendations"])
except KeyError:
    print("No apartments meet this criteria")

print("\nOther apartments:")
try:
    display(results_2["other_apartments"].head(20))
except KeyError:
    print("No apartments meet this criteria")

/Users/ariellerabinovich/anaconda3/envs/ds_new/lib/python3.11/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)



Top 10 recommendations:


,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,water_included,listing_description,review_1,review_2,review_3,click_through_rate,combined_text,text_similarity,ctr_norm,final_score
0,APT-1026,Brighton,2754,3,3,1258,2,1949,1,1,...,1,Impeccably maintained three-bedroom in a sough...,Really enjoyed my time here. natural light and...,Very happy with this place. closet space is ex...,Very happy with this place. updated appliances...,0.0683,Impeccably maintained three-bedroom in a sough...,0.029998,1.000000,0.175498
1,APT-1234,Back Bay,4242,3,3,1302,5,1927,0,1,...,0,Beautifully renovated three-bedroom with moder...,Super comfortable space. rooftop access and I ...,Great apartment. central air made it feel like...,Had a great experience. gym and it's super wal...,0.0585,Beautifully renovated three-bedroom with moder...,0.038472,0.717579,0.140338
2,APT-1124,Brighton,3011,4,4,1563,3,1952,0,1,...,1,Cozy four-bedroom with good natural light and ...,It served its purpose. layout but management c...,Would rent again in a heartbeat. natural light...,Not great. a malfunctioning thermostat was nev...,0.0336,Cozy four-bedroom with good natural light and ...,0.004825,0.000000,0.004101



Other apartments:


,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,water_included,listing_description,review_1,review_2,review_3,click_through_rate,combined_text,text_similarity,ctr_norm,final_score


In [47]:
test_df_2

,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,gym_in_building,balcony,pets_allowed,heat_included,water_included,listing_description,review_1,review_2,review_3,click_through_rate
26,APT-1026,Brighton,2754,3,3,1258,2,1949,1,1,...,1,0,1,0,1,Impeccably maintained three-bedroom in a sough...,Really enjoyed my time here. natural light and...,Very happy with this place. closet space is ex...,Very happy with this place. updated appliances...,0.0683
124,APT-1124,Brighton,3011,4,4,1563,3,1952,0,1,...,0,0,1,1,1,Cozy four-bedroom with good natural light and ...,It served its purpose. layout but management c...,Would rent again in a heartbeat. natural light...,Not great. a malfunctioning thermostat was nev...,0.0336
234,APT-1234,Back Bay,4242,3,3,1302,5,1927,0,1,...,1,0,0,1,0,Beautifully renovated three-bedroom with moder...,Super comfortable space. rooftop access and I ...,Great apartment. central air made it feel like...,Had a great experience. gym and it's super wal...,0.0585


For this prompt, we can see the classic way of filtering recommended an apartment with worse reviews 2nd, whereas our bot recommended this apartment last.

In [48]:
test_prompt_3

"I want a 2 bedroom 2 bath in Beacon Hill. I don't need A/C but would prefer somewhere over 650 sqft.My roommate and I want to be close to a gym and grocery store. I don't care about furnishings or appliances but     I want there to be tons of natural light, as I want it to feel homey. I don't need heat/water included."

In [50]:
results_3= recommend_apartments_llm(apartments_df, test_prompt_3, generator, top_n=5)
print("\nTop 10 recommendations:")
try:
    display(results_3["top_10_recommendations"])
except KeyError:
    print("No apartments meet this criteria")

print("\nOther apartments:")
try:
    display(results_3["other_apartments"].head(20))
except KeyError:
    print("No apartments meet this criteria")

/Users/ariellerabinovich/anaconda3/envs/ds_new/lib/python3.11/site-packages/transformers/pytorch_utils.py:339: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_elements = torch.tensor(test_elements)



Top 10 recommendations:


,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,water_included,listing_description,review_1,review_2,review_3,click_through_rate,combined_text,text_similarity,ctr_norm,final_score
0,APT-1359,Beacon Hill,3959,2,2,1073,6,1911,1,1,...,1,Move-in ready two-bedroom with designer touche...,Lived here for two years and loved it. rooftop...,Would rent again in a heartbeat. hardwood floo...,Adequate for the area. bay windows. Could use ...,0.0786,Move-in ready two-bedroom with designer touche...,0.011119,1.000000,0.159451
1,APT-1418,Beacon Hill,3900,2,2,800,3,1920,0,1,...,1,Elegant two-bedroom offering a perfect blend o...,The apartment exceeded my expectations. quiet ...,Great apartment. rooftop access made it feel l...,Great apartment. natural light made it feel li...,0.0560,Elegant two-bedroom offering a perfect blend o...,0.045846,0.429293,0.103363
2,APT-1148,Beacon Hill,5134,3,3,1248,4,1933,1,1,...,0,Stunning three-bedroom in the heart of Beacon ...,"Clean, well-maintained, and courtyard. No comp...",Solid place to live. modern finishes was bette...,My favorite apartment so far. central air and ...,0.0535,Stunning three-bedroom in the heart of Beacon ...,0.047924,0.366162,0.095660
3,APT-1252,Beacon Hill,5414,3,3,1122,1,1928,0,1,...,0,Beautifully renovated three-bedroom with moder...,Really appreciated high ceilings. The manageme...,Really enjoyed my time here. central air and t...,Average experience overall. renovated bathroom...,0.0580,Beautifully renovated three-bedroom with moder...,0.003614,0.479798,0.075042
4,APT-1276,Beacon Hill,4509,2,2,787,1,1904,0,1,...,0,Elegant two-bedroom offering a perfect blend o...,Really enjoyed my time here. rooftop access an...,One of the better apartments I've rented. cent...,Perfect for young professionals. storage space...,0.0390,Elegant two-bedroom offering a perfect blend o...,0.034443,0.000000,0.029276



Other apartments:


,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,water_included,listing_description,review_1,review_2,review_3,click_through_rate,combined_text,text_similarity,ctr_norm,final_score


In [51]:
test_df_3

,listing_id,neighborhood,rent,bedrooms,bathrooms,sqft,floor,year_built,in_unit_laundry,dishwasher,...,gym_in_building,balcony,pets_allowed,heat_included,water_included,listing_description,review_1,review_2,review_3,click_through_rate
148,APT-1148,Beacon Hill,5134,3,3,1248,4,1933,1,1,...,1,0,0,1,0,Stunning three-bedroom in the heart of Beacon ...,"Clean, well-maintained, and courtyard. No comp...",Solid place to live. modern finishes was bette...,My favorite apartment so far. central air and ...,0.0535
252,APT-1252,Beacon Hill,5414,3,3,1122,1,1928,0,1,...,1,0,0,0,0,Beautifully renovated three-bedroom with moder...,Really appreciated high ceilings. The manageme...,Really enjoyed my time here. central air and t...,Average experience overall. renovated bathroom...,0.0580
276,APT-1276,Beacon Hill,4509,2,2,787,1,1904,0,1,...,1,0,1,0,0,Elegant two-bedroom offering a perfect blend o...,Really enjoyed my time here. rooftop access an...,One of the better apartments I've rented. cent...,Perfect for young professionals. storage space...,0.0390
359,APT-1359,Beacon Hill,3959,2,2,1073,6,1911,1,1,...,1,0,1,0,1,Move-in ready two-bedroom with designer touche...,Lived here for two years and loved it. rooftop...,Would rent again in a heartbeat. hardwood floo...,Adequate for the area. bay windows. Could use ...,0.0786
418,APT-1418,Beacon Hill,3900,2,2,800,3,1920,0,1,...,1,0,1,0,1,Elegant two-bedroom offering a perfect blend o...,The apartment exceeded my expectations. quiet ...,Great apartment. rooftop access made it feel l...,Great apartment. natural light made it feel li...,0.0560


Our bot recommended the two apartments with good natural lighting and rooftop access first (following the user's requests), whicle the classic way recommended these towards the bottom (3rd and 5th). 